# External Validation — Leave-One-Hospital-Out

**Why this notebook exists:** every result so far (`03_Baseline_Models.ipynb`,
`04_All_Models_Comparison.ipynb`) uses a random train/test split drawn from *all four hospitals
pooled together*. That tells us how well the model does on patients from the same mix of
hospitals it was trained on — it does **not** tell us how well the model would generalize to a
genuinely new hospital it has never seen data from, which is a much stronger and more honest test.

This notebook trains on **three** of the four hospital sources and tests purely on the **fourth,
completely held-out** source, rotating through all four. This is called **Leave-One-Group-Out**
validation.

**Important caveat found while building this:** the four sources have very different class
balances —

| Source | # rows | % positive (heart disease) |
|---|---|---|
| Cleveland | 303 | 45.9% |
| Hungarian | 294 | 36.1% |
| VA | 200 | 74.5% |
| Switzerland | 123 | **93.5%** |

Switzerland in particular is almost entirely positive cases. That means a trivial "always predict
disease" model would score ~93.5% accuracy on Switzerland alone — so for held-out sources like
this, **accuracy is a misleading metric on its own**; we report **balanced accuracy** and
**ROC-AUC** alongside it, since those account for class imbalance.

## 1. Load Data

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, f1_score

df = pd.read_csv("../data/raw/heart2.csv").drop_duplicates().reset_index(drop=True)

numeric_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical_cols = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

df["source"].value_counts()

source
cleveland      303
hungarian      293
va             199
switzerland    123
Name: count, dtype: int64

## 2. Class Balance Per Source

Confirms the imbalance noted above before we interpret any results.

In [2]:
df.groupby("source")["target"].value_counts(normalize=True).unstack().round(3)

target,0,1
source,,
cleveland,0.541,0.459
hungarian,0.638,0.362
switzerland,0.065,0.935
va,0.256,0.744


## 3. Leave-One-Hospital-Out Evaluation Function

For each source: train on the other three (with the same leakage-free pipeline as before — IQR
bounds, imputer, and scaler all fit only on the training portion), then test purely on the
held-out source.

In [3]:
def evaluate_leave_one_out(model_builder, model_name):
    sources = df["source"].unique()
    rows = []

    for held_out in sources:
        train_df = df[df["source"] != held_out]
        test_df = df[df["source"] == held_out]

        X_train = train_df[numeric_cols + categorical_cols].copy()
        y_train = train_df["target"]
        X_test = test_df[numeric_cols + categorical_cols].copy()
        y_test = test_df["target"]

        # IQR bounds learned from training sources only
        for col in numeric_cols:
            Q1, Q3 = X_train[col].quantile(0.25), X_train[col].quantile(0.75)
            IQR = Q3 - Q1
            lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
            X_train[col] = X_train[col].clip(lower, upper)
            X_test[col] = X_test[col].clip(lower, upper)

        pre = ColumnTransformer([
            ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_cols),
            ("cat", SimpleImputer(strategy="most_frequent"), categorical_cols)
        ])
        X_train_p = pre.fit_transform(X_train)
        X_test_p = pre.transform(X_test)

        model = model_builder()
        model.fit(X_train_p, y_train)
        pred = model.predict(X_test_p)
        proba = model.predict_proba(X_test_p)[:, 1]

        rows.append({
            "Model": model_name,
            "Held-out source": held_out,
            "Test rows": len(test_df),
            "% positive in held-out": round(y_test.mean() * 100, 1),
            "Accuracy": accuracy_score(y_test, pred),
            "Balanced Accuracy": balanced_accuracy_score(y_test, pred),
            "ROC-AUC": roc_auc_score(y_test, proba),
            "F1": f1_score(y_test, pred)
        })

    return pd.DataFrame(rows)

## 4. Run It — SVM (our current best model on pooled data)

In [4]:
svm_results = evaluate_leave_one_out(
    lambda: SVC(probability=True, random_state=42, C=10, gamma=0.01),
    "SVM"
)
svm_results

,Model,Held-out source,Test rows,% positive in held-out,Accuracy,Balanced Accuracy,ROC-AUC,F1
0,SVM,cleveland,303,45.9,0.785479,0.787024,0.862432,0.775087
1,SVM,hungarian,293,36.2,0.822526,0.818056,0.876476,0.765766
2,SVM,switzerland,123,93.5,0.796748,0.775000,0.710870,0.880383
3,SVM,va,199,74.4,0.698492,0.662361,0.703498,0.784173


## 5. Run It — Random Forest (for comparison)

In [5]:
rf_results = evaluate_leave_one_out(
    lambda: RandomForestClassifier(random_state=42, n_estimators=200),
    "Random Forest"
)
rf_results

,Model,Held-out source,Test rows,% positive in held-out,Accuracy,Balanced Accuracy,ROC-AUC,F1
0,Random Forest,cleveland,303,45.9,0.778878,0.778733,0.859844,0.763251
1,Random Forest,hungarian,293,36.2,0.808874,0.815533,0.875164,0.760684
2,Random Forest,switzerland,123,93.5,0.699187,0.664674,0.696739,0.814070
3,Random Forest,va,199,74.4,0.673367,0.632618,0.673225,0.765343


## 6. Combined View & Averages

In [6]:
all_results = pd.concat([svm_results, rf_results], ignore_index=True)
all_results

,Model,Held-out source,Test rows,% positive in held-out,Accuracy,Balanced Accuracy,ROC-AUC,F1
0,SVM,cleveland,303,45.9,0.785479,0.787024,0.862432,0.775087
1,SVM,hungarian,293,36.2,0.822526,0.818056,0.876476,0.765766
2,SVM,switzerland,123,93.5,0.796748,0.775000,0.710870,0.880383
3,SVM,va,199,74.4,0.698492,0.662361,0.703498,0.784173
4,Random Forest,cleveland,303,45.9,0.778878,0.778733,0.859844,0.763251
5,Random Forest,hungarian,293,36.2,0.808874,0.815533,0.875164,0.760684
6,Random Forest,switzerland,123,93.5,0.699187,0.664674,0.696739,0.814070
7,Random Forest,va,199,74.4,0.673367,0.632618,0.673225,0.765343


In [7]:
summary = all_results.groupby("Model")[["Accuracy", "Balanced Accuracy", "ROC-AUC", "F1"]].mean().round(3)
summary

,Accuracy,Balanced Accuracy,ROC-AUC,F1
Model,,,,
Random Forest,0.740,0.723,0.776,0.776
SVM,0.776,0.761,0.788,0.801
